# Bulk Backtest — E1 & E2

Run backtests across all pairs with parquet data, inspect results visually, then push verdicts to MongoDB `pair_historical`.

**Workflow:**
1. Select engine (E1 or E2)
2. Discover pairs from parquet cache
3. Run backtests (BacktestingEngine)
4. Summary table + verdict distribution
5. Drill into individual pairs (equity curves, trade scatter)
6. Push verdicts to MongoDB

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, '/Users/hermes/quants-lab')

import datetime
import asyncio
from decimal import Decimal
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.backtesting import BacktestingEngine
from core.data_paths import data_paths

backtesting = BacktestingEngine(load_cached_data=True)
print(f"BacktestingEngine ready — candles dir: {data_paths.candles_dir}")

## 1. Configuration

Switch `ENGINE` between `"E1"` and `"E2"`. Everything else adapts automatically.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CHANGE THIS TO SWITCH ENGINE                    ║
# ╚══════════════════════════════════════════════════╝
ENGINE = "E1"  # "E1" or "E2"

# Backtest parameters
CONNECTOR = "bybit_perpetual"
BACKTEST_DAYS = 365   # full year for statistical significance
TRADE_COST = 0.000375  # 0.02% maker + 0.055% taker = 0.075% RT

# Verdict thresholds
PF_ALLOW = 1.3
PF_WATCH = 1.0
MIN_TRADES = 10

# Engine registry handles resolution, candles, exit params, config class
from app.engines.registry import get_engine, build_backtest_config

engine_meta = get_engine(ENGINE)
RESOLUTION = engine_meta["backtesting_resolution"]


def make_config(pair: str):
    """Build config via engine registry — handles everything automatically."""
    return build_backtest_config(engine_name=ENGINE, connector=CONNECTOR, pair=pair)


# Quick test
test_config = make_config("BTC-USDT")
print(f"Engine: {ENGINE} ({engine_meta['name']}) | Architecture: {engine_meta['architecture']}")
print(f"Resolution: {RESOLUTION} | Days: {BACKTEST_DAYS} | Cost: {TRADE_COST}")
print(f"Candles config: {[(c.interval, c.max_records) for c in test_config.candles_config]}")
print(f"Exit: SL={engine_meta['exit_params']['stop_loss']} TP={engine_meta['exit_params']['take_profit']} TL={engine_meta['exit_params']['time_limit']}s")
est_steps = BACKTEST_DAYS * 24 * (12 if RESOLUTION == "5m" else 1)
print(f"Estimated steps per pair: ~{est_steps:,}")

## 2. Discover Pairs

Scan parquet cache for all pairs with 1h data. Shows how many bars each pair has.

In [ ]:
candles_dir = data_paths.candles_dir
min_bars = BACKTEST_DAYS * 24  # need at least this many 1h bars

pair_info = []
for f in sorted(candles_dir.glob(f"{CONNECTOR}|*|1h.parquet")):
    parts = f.stem.split("|")
    if len(parts) == 3:
        pair = parts[1]
        df = pd.read_parquet(f)
        n_bars = len(df)
        pair_info.append({"pair": pair, "bars_1h": n_bars, "file": f.name})

pair_df = pd.DataFrame(pair_info).sort_values("bars_1h", ascending=False)
eligible = pair_df[pair_df["bars_1h"] >= min_bars]["pair"].tolist()

print(f"Found {len(pair_df)} pairs with 1h parquet data")
print(f"Eligible (>= {min_bars} bars): {len(eligible)} pairs")
print(f"Skipped (insufficient data): {len(pair_df) - len(eligible)}")
pair_df.head(10)

## 3. Run Bulk Backtests

Loops over all eligible pairs. Stores `BacktestingResult` objects for later drill-down.

In [ ]:
now_ts = int(datetime.datetime.now(datetime.timezone.utc).timestamp())
start_ts = now_ts - BACKTEST_DAYS * 86400

period_label = (
    f"{datetime.datetime.utcfromtimestamp(start_ts).strftime('%Y-%m-%d')}"
    f"_{datetime.datetime.utcfromtimestamp(now_ts).strftime('%Y-%m-%d')}"
)
print(f"Backtest period: {period_label}\n")

results = []          # summary rows
bt_results = {}       # pair -> BacktestingResult (for drill-down)

for i, pair in enumerate(eligible):
    try:
        # Fresh engine per pair — reuse causes state corruption (known HB bug)
        bt_engine = BacktestingEngine(load_cached_data=True)
        config = make_config(pair)
        bt_result = await bt_engine.run_backtesting(
            config=config,
            start=start_ts,
            end=now_ts,
            backtesting_resolution=RESOLUTION,
            trade_cost=TRADE_COST,
        )
        # Workaround: close_types can be int(0) when no trades
        if not isinstance(bt_result.results.get("close_types"), dict):
            bt_result.results["close_types"] = {}

        r = bt_result.results
        trades = r.get("total_executors", 0)
        pf = r.get("profit_factor", 0) or 0
        wr = (r.get("accuracy_long", 0) or 0) * 100  # as percentage
        sharpe = r.get("sharpe_ratio", None)
        max_dd = r.get("max_drawdown_pct", None)
        pnl = r.get("net_pnl_quote", 0) or 0

        # Compute verdict
        if trades < MIN_TRADES:
            verdict = "BLOCK"
        elif pf >= PF_ALLOW:
            verdict = "ALLOW"
        elif pf >= PF_WATCH:
            verdict = "WATCH"
        else:
            verdict = "BLOCK"

        results.append({
            "pair": pair,
            "trades": trades,
            "pf": round(pf, 2),
            "win_rate": round(wr, 1),
            "sharpe": round(sharpe, 2) if sharpe else None,
            "max_dd_pct": round(max_dd * 100, 1) if max_dd else None,
            "pnl_quote": round(pnl, 2),
            "n_long": r.get("total_long", 0),
            "n_short": r.get("total_short", 0),
            "close_types": r.get("close_types", {}),
            "verdict": verdict,
        })
        bt_results[pair] = bt_result

        marker = {"ALLOW": "+", "WATCH": "~", "BLOCK": "x"}[verdict]
        print(f"  [{marker}] {pair:>12s}  PF={pf:5.2f}  WR={wr:4.1f}%  trades={trades:3d}  PnL={pnl:+8.2f}  -> {verdict}")

    except Exception as e:
        print(f"  [!] {pair:>12s}  ERROR: {e}")
        results.append({"pair": pair, "trades": 0, "pf": 0, "verdict": "ERROR", "error": str(e)})

print(f"\nDone: {len(results)} pairs processed, {len(bt_results)} successful")

## Debug: Single-Pair Signal Inspection

Run one pair to check if the controller generates any signals at all.

In [ ]:
# Entry quality gate sensitivity test
# Compare: locked (0.3/0.5/5bps) vs relaxed (0.5/0.8/no gap) vs wide (1.0/1.0/no gap) vs disabled
# Per research-process skill: structural fix BEFORE Optuna. One variable at a time.

import traceback
from app.engines.registry import build_backtest_config

# Pairs that worked in prior run (skip HB cache bug pairs)
TEST_PAIRS = ["BTC-USDT", "ETH-USDT", "SOL-USDT", "ADA-USDT", "NEAR-USDT", "AVAX-USDT"]

now_ts = int(datetime.datetime.now(datetime.timezone.utc).timestamp())
start_ts = now_ts - BACKTEST_DAYS * 86400

VARIANTS = {
    "locked":   dict(),  # defaults: 0.3 ATR distance, 0.5 ATR body, 5 bps gap
    "relaxed":  dict(entry_distance_atr_mult=0.5, entry_body_atr_mult=0.8, entry_gap_bps_floor=20.0),
    "wide":     dict(entry_distance_atr_mult=1.0, entry_body_atr_mult=1.0, entry_gap_bps_floor=50.0),
    "disabled": dict(entry_quality_filter=False),
}

all_results = []

for variant_name, overrides in VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"VARIANT: {variant_name}  {overrides or '(defaults)'}")
    print(f"{'='*60}")

    variant_trades = 0
    variant_pnl = 0
    variant_signals = 0

    for pair in TEST_PAIRS:
        try:
            bt = BacktestingEngine(load_cached_data=True)
            config = build_backtest_config(
                engine_name=ENGINE, connector=CONNECTOR, pair=pair, **overrides
            )
            result = await bt.run_backtesting(
                config=config, start=start_ts, end=now_ts,
                backtesting_resolution=RESOLUTION, trade_cost=TRADE_COST,
            )
            if not isinstance(result.results.get("close_types"), dict):
                result.results["close_types"] = {}

            r = result.results
            trades = r.get("total_executors", 0)
            pf = r.get("profit_factor", 0) or 0
            pnl = r.get("net_pnl_quote", 0) or 0
            wr = (r.get("accuracy_long", 0) or 0) * 100

            pdf = result.processed_data
            signals = int((pdf["signal"] != 0).sum()) if "signal" in pdf.columns else 0
            eq_pass = int((pdf["entry_quality_pass"] != 0).sum()) if "entry_quality_pass" in pdf.columns else 0

            variant_trades += trades
            variant_pnl += pnl
            variant_signals += signals

            all_results.append({
                "variant": variant_name, "pair": pair,
                "signals": signals, "eq_pass": eq_pass, "trades": trades,
                "pf": round(pf, 2), "wr": round(wr, 1), "pnl": round(pnl, 2),
            })

            marker = "+" if pf >= 1.3 and trades >= 5 else ("~" if trades > 0 else "x")
            print(f"  [{marker}] {pair:>12s}  sig={signals:3d}  trades={trades:3d}  PF={pf:6.2f}  WR={wr:5.1f}%  PnL={pnl:+8.2f}")

        except Exception as e:
            print(f"  [!] {pair:>12s}  ERROR: {e}")

    print(f"  --- TOTAL: {variant_signals} signals, {variant_trades} trades, PnL={variant_pnl:+.2f}")

# Summary comparison
print(f"\n\n{'='*70}")
print("VARIANT COMPARISON")
print(f"{'='*70}")
comp_df = pd.DataFrame(all_results)
summary = comp_df.groupby("variant").agg(
    pairs=("pair", "count"),
    total_signals=("signals", "sum"),
    total_trades=("trades", "sum"),
    total_pnl=("pnl", "sum"),
    avg_pf=("pf", "mean"),
    avg_wr=("wr", "mean"),
).reindex(["locked", "relaxed", "wide", "disabled"])
print(summary.to_string())
print(f"\nSignal multiplier vs locked:")
locked_sig = summary.loc["locked", "total_signals"]
for v in ["relaxed", "wide", "disabled"]:
    mult = summary.loc[v, "total_signals"] / max(locked_sig, 1)
    print(f"  {v}: {mult:.1f}x signals")

## 4. Summary Table

All pairs sorted by profit factor. Color-coded by verdict.

In [ ]:
summary_df = pd.DataFrame(results).sort_values("pf", ascending=False)
display_cols = ["pair", "verdict", "pf", "win_rate", "sharpe", "trades", "pnl_quote", "max_dd_pct", "n_long", "n_short"]
display_df = summary_df[[c for c in display_cols if c in summary_df.columns]].reset_index(drop=True)

# Verdict counts
counts = summary_df["verdict"].value_counts()
print(f"ALLOW: {counts.get('ALLOW', 0)}  |  WATCH: {counts.get('WATCH', 0)}  |  BLOCK: {counts.get('BLOCK', 0)}  |  ERROR: {counts.get('ERROR', 0)}")
print(f"Total PnL (ALLOW pairs): {summary_df[summary_df['verdict'] == 'ALLOW']['pnl_quote'].sum():.2f} USDT")
print()

# Style the dataframe
def color_verdict(val):
    if val == "ALLOW":
        return "background-color: #1a472a; color: #00ff88"
    elif val == "WATCH":
        return "background-color: #4a3f00; color: #ffd700"
    elif val == "BLOCK":
        return "background-color: #4a1a1a; color: #ff6666"
    return ""

display_df.style.map(color_verdict, subset=["verdict"])

## 5. Verdict Distribution & PF Scatter

In [ ]:
valid_df = summary_df[summary_df["verdict"] != "ERROR"].copy()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Verdict Distribution", "Profit Factor vs Win Rate"),
                    column_widths=[0.35, 0.65])

# Left: verdict bar chart
color_map = {"ALLOW": "#00ff88", "WATCH": "#ffd700", "BLOCK": "#ff6666"}
vc = valid_df["verdict"].value_counts().reindex(["ALLOW", "WATCH", "BLOCK"]).fillna(0)
fig.add_trace(go.Bar(
    x=vc.index, y=vc.values,
    marker_color=[color_map[v] for v in vc.index],
    text=vc.values, textposition="auto",
), row=1, col=1)

# Right: PF vs WR scatter (size = trades)
fig.add_trace(go.Scatter(
    x=valid_df["win_rate"], y=valid_df["pf"],
    mode="markers+text",
    text=valid_df["pair"].str.replace("-USDT", ""),
    textposition="top center",
    textfont=dict(size=9, color="white"),
    marker=dict(
        size=np.clip(valid_df["trades"], 5, 60),
        color=[color_map[v] for v in valid_df["verdict"]],
        opacity=0.8,
        line=dict(width=1, color="white"),
    ),
    hovertext=[f"{r['pair']}<br>PF={r['pf']}<br>WR={r['win_rate']}%<br>Trades={r['trades']}<br>PnL={r['pnl_quote']}"
               for _, r in valid_df.iterrows()],
    hoverinfo="text",
    showlegend=False,
), row=1, col=2)

# Reference lines
fig.add_hline(y=PF_ALLOW, line_dash="dash", line_color="#00ff88", annotation_text=f"ALLOW (PF>={PF_ALLOW})", row=1, col=2)
fig.add_hline(y=PF_WATCH, line_dash="dash", line_color="#ffd700", annotation_text=f"WATCH (PF>={PF_WATCH})", row=1, col=2)

fig.update_layout(
    title=f"{ENGINE} Bulk Backtest — {period_label}",
    height=500, showlegend=False,
    plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
    font=dict(color="white"),
)
fig.update_xaxes(gridcolor="gray")
fig.update_yaxes(gridcolor="gray")
fig.show()

## 6. Cumulative PnL — ALLOW Pairs

Equity curves for all pairs that passed the ALLOW threshold. Useful for spotting regime sensitivity.

In [ ]:
allow_pairs = summary_df[summary_df["verdict"] == "ALLOW"]["pair"].tolist()

if not allow_pairs:
    print("No ALLOW pairs found. Try lowering PF_ALLOW or check if the engine generates trades.")
else:
    fig = go.Figure()
    for pair in allow_pairs:
        bt = bt_results[pair]
        edf = bt.executors_df.sort_values("close_timestamp").copy()
        edf["cum_pnl"] = edf["net_pnl_quote"].cumsum()
        fig.add_trace(go.Scatter(
            x=edf["close_timestamp"], y=edf["cum_pnl"],
            mode="lines", name=pair.replace("-USDT", ""),
            line=dict(width=2),
        ))

    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"{ENGINE} Cumulative PnL — ALLOW Pairs ({len(allow_pairs)})",
        yaxis_title="Cumulative PnL (USDT)", height=450,
        plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
        font=dict(color="white"),
        xaxis=dict(gridcolor="gray"), yaxis=dict(gridcolor="gray"),
        legend=dict(bgcolor="rgba(0,0,0,0.5)"),
    )
    fig.show()

## 7. Exit Type Breakdown — All Pairs

Aggregated view of how trades close across all pairs. Helps spot systematic issues (e.g. too many TIME_LIMIT = bad sizing).

In [ ]:
# Aggregate close_types across all pairs
from collections import Counter

total_exits = Counter()
for row in results:
    ct = row.get("close_types", {})
    if isinstance(ct, dict):
        for k, v in ct.items():
            label = k.name if hasattr(k, "name") else str(k)
            total_exits[label] += v

if total_exits:
    exit_df = pd.DataFrame(list(total_exits.items()), columns=["close_type", "count"]).sort_values("count", ascending=False)
    fig = px.bar(exit_df, x="close_type", y="count", title=f"{ENGINE} Exit Types (all pairs)",
                 color="close_type", color_discrete_sequence=px.colors.qualitative.Set2)
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
        font=dict(color="white"), showlegend=False,
    )
    fig.show()
    print(exit_df.to_string(index=False))
else:
    print("No trades found across any pair.")

## 8. Drill Into a Single Pair

Change `DRILL_PAIR` to inspect any pair in detail. Uses QL native `get_backtesting_figure()` (candlestick + entries/exits + cum PnL) plus trade scatter and PnL distribution.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CHANGE THIS TO DRILL INTO A DIFFERENT PAIR      ║
# ╚══════════════════════════════════════════════════╝
DRILL_PAIR = allow_pairs[0] if allow_pairs else eligible[0]

if DRILL_PAIR not in bt_results:
    print(f"{DRILL_PAIR} not in results — pick from: {list(bt_results.keys())[:10]}")
else:
    bt = bt_results[DRILL_PAIR]
    row = next(r for r in results if r["pair"] == DRILL_PAIR)
    print(f"=== {DRILL_PAIR} ({ENGINE}) — {row['verdict']} ===")
    print(bt.get_results_summary())

In [ ]:
# QL native backtesting figure — candlestick + entry/exit traces + cumulative PnL
if DRILL_PAIR in bt_results:
    fig = bt_results[DRILL_PAIR].get_backtesting_figure()
    fig.update_layout(title=f"{ENGINE} — {DRILL_PAIR} — Entries & Exits", height=700)
    fig.show()

In [ ]:
# PnL scatter + distribution for the drilled pair
if DRILL_PAIR in bt_results:
    edf = bt_results[DRILL_PAIR].executors_df.copy()
    if len(edf) > 0:
        edf["profitable"] = edf["net_pnl_quote"] > 0

        fig = make_subplots(rows=1, cols=2, subplot_titles=("PnL per Trade", "PnL Distribution"))

        # Scatter
        for is_profit, color in [(True, "#00ff88"), (False, "#ff6666")]:
            subset = edf[edf["profitable"] == is_profit]
            fig.add_trace(go.Scatter(
                x=subset["timestamp"], y=subset["net_pnl_quote"],
                mode="markers", marker=dict(color=color, size=8, opacity=0.8),
                name="Win" if is_profit else "Loss",
            ), row=1, col=1)
        fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

        # Histogram
        fig.add_trace(go.Histogram(
            x=edf["net_pnl_quote"], nbinsx=25,
            marker_color="#4488ff", opacity=0.7, name="PnL dist",
        ), row=1, col=2)
        fig.add_vline(x=0, line_dash="dash", line_color="white", row=1, col=2)

        fig.update_layout(
            title=f"{DRILL_PAIR} — Trade Analysis",
            height=400, showlegend=False,
            plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
            font=dict(color="white"),
        )
        fig.update_xaxes(gridcolor="gray")
        fig.update_yaxes(gridcolor="gray")
        fig.show()

        # Exit type breakdown
        edf["close_type_str"] = edf["close_type"].apply(lambda x: x.name if hasattr(x, "name") else str(x))
        print(edf[["side", "net_pnl_quote", "close_type_str"]].groupby(["side", "close_type_str"]).agg(
            count=("net_pnl_quote", "count"), avg_pnl=("net_pnl_quote", "mean")
        ).round(2))
    else:
        print(f"No trades for {DRILL_PAIR}")

## 9. WATCH Pairs — Borderline Candidates

Pairs just below the ALLOW threshold. Review these to decide if any should be promoted (e.g. with regime filtering).

In [ ]:
watch_pairs = summary_df[summary_df["verdict"] == "WATCH"]["pair"].tolist()

if not watch_pairs:
    print("No WATCH pairs.")
else:
    fig = go.Figure()
    for pair in watch_pairs:
        if pair in bt_results:
            bt = bt_results[pair]
            edf = bt.executors_df.sort_values("close_timestamp").copy()
            if len(edf) > 0:
                edf["cum_pnl"] = edf["net_pnl_quote"].cumsum()
                fig.add_trace(go.Scatter(
                    x=edf["close_timestamp"], y=edf["cum_pnl"],
                    mode="lines", name=pair.replace("-USDT", ""),
                    line=dict(width=2, dash="dot"),
                ))

    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"{ENGINE} Cumulative PnL — WATCH Pairs ({len(watch_pairs)})",
        yaxis_title="Cumulative PnL (USDT)", height=400,
        plot_bgcolor='rgba(0,0,0,0.85)', paper_bgcolor='rgba(0,0,0,0.85)',
        font=dict(color="white"),
        xaxis=dict(gridcolor="gray"), yaxis=dict(gridcolor="gray"),
        legend=dict(bgcolor="rgba(0,0,0,0.5)"),
    )
    fig.show()

    watch_df = summary_df[summary_df["verdict"] == "WATCH"][display_cols].reset_index(drop=True)
    display(watch_df)

## 10. Push Verdicts to MongoDB

Writes results to `pair_historical` with the same schema as `BulkBacktestTask`. Run this cell only after you've reviewed the results above and are satisfied.

**This overwrites existing verdicts for this engine.**

In [ ]:
import os
from motor.motor_asyncio import AsyncIOMotorClient

MONGO_URI = os.environ.get("MONGO_URI", "mongodb://localhost:27017/quants_lab")
MONGO_DB = os.environ.get("MONGO_DATABASE", "quants_lab")

client = AsyncIOMotorClient(MONGO_URI)
db = client[MONGO_DB]
collection = db["pair_historical"]

pushed = 0
for row in results:
    if row.get("verdict") == "ERROR":
        continue

    doc = {
        "engine": ENGINE,
        "pair": row["pair"],
        "period": period_label,
        "trades": row["trades"],
        "profit_factor": row["pf"],
        "win_rate": row["win_rate"],
        "pnl_quote": row["pnl_quote"],
        "max_dd_pct": row.get("max_dd_pct"),
        "sharpe": row.get("sharpe"),
        "n_long": row.get("n_long", 0),
        "n_short": row.get("n_short", 0),
        "verdict": row["verdict"],
        "created_at": int(datetime.datetime.now(datetime.timezone.utc).timestamp() * 1000),
    }
    await collection.update_one(
        {"engine": ENGINE, "pair": row["pair"]},
        {"$set": doc},
        upsert=True,
    )
    pushed += 1

print(f"Pushed {pushed} verdicts to pair_historical for {ENGINE}")
print(f"  ALLOW: {sum(1 for r in results if r['verdict'] == 'ALLOW')}")
print(f"  WATCH: {sum(1 for r in results if r['verdict'] == 'WATCH')}")
print(f"  BLOCK: {sum(1 for r in results if r['verdict'] == 'BLOCK')}")

## 11. Verify — Read Back from MongoDB

In [ ]:
cursor = collection.find({"engine": ENGINE}, {"_id": 0}).sort("profit_factor", -1)
docs = await cursor.to_list(length=200)
mongo_df = pd.DataFrame(docs)

if len(mongo_df) > 0:
    print(f"pair_historical has {len(mongo_df)} entries for {ENGINE}:")
    print(f"  ALLOW: {(mongo_df['verdict'] == 'ALLOW').sum()}")
    print(f"  WATCH: {(mongo_df['verdict'] == 'WATCH').sum()}")
    print(f"  BLOCK: {(mongo_df['verdict'] == 'BLOCK').sum()}")
    print()
    display(mongo_df[["pair", "verdict", "profit_factor", "win_rate", "trades", "pnl_quote", "sharpe"]].head(15))
else:
    print("No entries found — did you run the push cell above?")